# Cách dựng luồng dữ liệu quyết định khoảng cách "mô hình cũ": làm lại từng bước và kiểm mọi con số

Bài này hỏi một câu về phương pháp đánh giá: khi dựng một luồng dữ liệu theo thời gian để đo một bộ phát hiện URL lừa đảo "già đi"
nhanh thế nào, thì bao nhiêu phần của khoảng cách giữa mô hình không cập nhật và mô hình được huấn luyện lại là do dữ liệu thật sự
thay đổi, và bao nhiêu phần là do cách dựng luồng? Trong PhishVN chỉ 41% số dòng có ngày; phần còn lại được rải đều theo hàm băm
trên khoảng thời gian có ngày. Notebook này làm lại toàn bộ thí nghiệm của bài từ bộ dữ liệu PhishVN đã công bố, trong một file duy
nhất. Mọi hàm đều nằm ngay trong notebook, nên đọc từ trên xuống là thấy hết cách mỗi con số được tính ra. Notebook chạy độc lập,
không cần repo nào khác.

### Cách chạy

1. Tải thư mục chứa notebook này (gồm `stream_construction_walkthrough.ipynb`, thư mục `expected/` và `requirements.txt`).
2. Cài thư viện đúng phiên bản: `pip install -r requirements.txt` (Python 3.12).
3. Mở notebook và chạy hết (Run All).

Lần chạy đầu, notebook tự tải gói `PhishVN_v3.1.0_open.zip` (khoảng 6 MB) từ bản 4 của bộ dữ liệu PhishVN trên Mendeley Data
(DOI 10.17632/b97hxbxtpd.4), kiểm SHA-256 của gói, rồi lấy ra hai file: `dataset_url.csv` (53.116 URL, nhãn, ngày thu thập, nguồn,
tầng nhãn) và `vn_compphish.csv` (21 đặc trưng từ vựng của mỗi URL, cùng thứ tự dòng). Bước 11 còn đọc bốn cột của PhreshPhish
từ Hugging Face tại một revision cố định (không tải cột HTML).

Mọi file tải về và mọi kết quả trung gian nằm trong thư mục `stream_work/` cạnh notebook. Cả notebook chạy khoảng 20 đến 30 phút
trên CPU, chưa kể thời gian tải; phần lâu nhất là lưới ngân sách gán nhãn lặp năm seed (Bước 8), mười cách rải lại theo hàm băm (Bước 9d) và hai mươi lần
gán nhãn sai ngẫu nhiên (Bước 10).

### Notebook kiểm gì

Mỗi thí nghiệm in kết quả ra màn hình và ghi lại đúng các bảng kết quả (CSV) và các file số liệu (JSON) mà bài đọc; notebook so
từng byte các file đó với bản kỳ vọng trong `expected/`. Với file JSON, notebook so thêm từng phần ngay khi phần đó tính xong, để
chỗ lệch hiện ra ở đúng bước gây ra nó. Notebook không vẽ hình.

Mỗi bước có phần giải thích, code, và dòng `[step] x/y checks match` ở cuối. Bước 12 gom tất cả lại. Rừng ngẫu nhiên, CatBoost và
SHAP đều chạy với seed cố định, nên kết quả giống hệt trên máy khác khi cài đúng phiên bản thư viện trong `requirements.txt`.

### Các bước

| Bước | Làm gì |
|---|---|
| 0 | thiết lập, tải dữ liệu |
| 1 | các hàm dùng chung: nạp luồng, rải dòng không ngày, PSI, mô hình, ba chính sách cập nhật |
| 2 | dựng luồng từ hai file dữ liệu |
| 3 | ba chính sách trên 20 cửa sổ, và hai kiến trúc tĩnh khác |
| 4 | năm bộ giám sát trôi dữ liệu, chấm theo một nhãn thay thế dựa trên kiểm định KS |
| 5 | biên ngân sách gán nhãn: ba mức ngân sách, rồi lưới dày bảy mức |
| 6 | độ nhạy theo seed của mô hình |
| 7 | thành phần từng cửa sổ, hai định dạng URL, và luồng chỉ gồm dòng có ngày |
| 8 | lưới ngân sách lặp trên năm seed |
| 9 | bốn cách dựng luồng từ cùng một corpus, đo bốn kiểu, và mười cách rải lại |
| 10 | độ nhạy với nhãn sai: đổi nhãn ngẫu nhiên 12,1% tin phishing tầng bronze |
| 11 | một corpus quốc tế có ngày ở mọi dòng (PhreshPhish): thứ tự ngày của nó cũng không phải thứ tự đến |
| 12 | tổng hợp |

### Thuật ngữ

| Từ | Nghĩa |
|---|---|
| luồng (stream) | toàn bộ dòng xếp theo thời gian rồi cắt thành 20 cửa sổ có số dòng bằng nhau |
| dòng rải (spread) | dòng không có ngày, được gán một ngày giả theo hàm băm SHA-1 của `id`, trải đều trên khoảng có ngày |
| static / periodic / drift | huấn luyện một lần ở cửa sổ 1; huấn luyện lại mỗi 3 cửa sổ; huấn luyện lại khi PSI vượt 0,2 hoặc F1 tụt quá 0,1 |
| AUTC | trung bình F1 theo cửa sổ, từ cửa sổ 2 đến 20 (diện tích dưới đường F1 theo thời gian) |
| khoảng cách (gap) | AUTC của chính sách cập nhật tốt hơn trừ AUTC của mô hình tĩnh |
| PSI | Population Stability Index: độ lệch giữa hai phân phối của một đặc trưng, tính trên 10 khoảng chia theo phân vị |
| KS | kiểm định Kolmogorov–Smirnov hai mẫu; ở đây dùng làm nhãn thay thế cho "cửa sổ có dịch chuyển" |
| ADWIN | bộ phát hiện thay đổi trên chuỗi lỗi của mô hình đang chạy (Bifet và Gavaldà, 2007) |
| SHAP-shift | khoảng cách biến phân toàn phần giữa hai vector tỷ trọng mean-\|SHAP\| của hai cửa sổ liên tiếp |
| ngân sách B | số nhãn tối đa được mua mỗi lần huấn luyện lại |

## Bước 0: Thiết lập và tải dữ liệu

Các thí nghiệm đọc và ghi theo một cây thư mục cố định (`data/processed/...`, `data/interim/...`). Notebook dựng đúng cây đó bên
trong `stream_work/`, đặt hai file dữ liệu vào `data/processed/`, rồi chuyển thư mục làm việc vào đó. Nhờ vậy code thí nghiệm giữ
nguyên đường dẫn gốc mà không ghi ra ngoài `stream_work/`.

Ô thứ hai định nghĩa các hàm kiểm:
- `check_file`: so từng byte một file notebook vừa ghi với bản kỳ vọng cùng tên;
- `check_json`: so một phần của file JSON kỳ vọng với giá trị notebook vừa tính (sau khi đi qua JSON, nên số thực được so chính xác
  đến từng bit, `NaN` được coi là bằng `NaN`);
- `write_results`: ghi một file JSON đúng định dạng của bản kỳ vọng (khóa sắp xếp, thụt 2 dấu cách).

In [ ]:
import os, sys, io, re, csv, json, math, time, hashlib, zipfile, warnings, filecmp, urllib.request
from collections import Counter
from math import log2
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

MENDELEY_ZIP = ("https://data.mendeley.com/public-files/datasets/b97hxbxtpd/files/"
                "d5724d4d-9c2b-46b7-810e-cf290c3bfd87/file_downloaded")
ZIP_SHA256 = "308351e9d0c0fca13a81f2f63524a7dee08ccba6c775848d125b3da5525d0ab5"
DATA_SHA256 = {"dataset_url.csv": "8e1c6325a82177014ac892a9aaf8c4d3cbbc986b47fd1ecac8899c865fb6eade",
               "vn_compphish.csv": "ce64567f4ab955b214b13a268546a460f2dd827b18432d0971b3f5a3ef06bbbf"}
EXPECTED_URL = "https://raw.githubusercontent.com/vuthainguyen1602/phishvn/master/notebooks/stream_construction/expected"

def fetch(url, dest):
    # Mendeley refuses Python's default user agent
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (PhishVN walkthrough notebook)"})
    Path(dest).write_bytes(urllib.request.urlopen(req, timeout=120).read())

HOME = Path.cwd()
WORK = HOME / "stream_work"
for d in ("data/processed/p5/dedup", "data/processed/p5/results", "data/processed/dedup_corpus", "data/interim"):
    (WORK / d).mkdir(parents=True, exist_ok=True)
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()

zip_path = WORK / "PhishVN_v3.1.0_open.zip"
if not zip_path.exists():
    fetch(MENDELEY_ZIP, zip_path)
assert sha(zip_path) == ZIP_SHA256, "the downloaded package is not the published one"
with zipfile.ZipFile(zip_path) as z:
    for name in DATA_SHA256:
        (WORK / "data/processed" / name).write_bytes(z.read(f"data/{name}"))
for name, h in DATA_SHA256.items():
    assert sha(WORK / "data/processed" / name) == h, name

EXPECTED = HOME / "expected"
EXPECTED_FILES = json.loads((EXPECTED / "manifest.json").read_text()) if (EXPECTED / "manifest.json").exists() else None
if EXPECTED_FILES is None:
    EXPECTED = WORK / "expected"
    EXPECTED.mkdir(exist_ok=True)
    fetch(f"{EXPECTED_URL}/manifest.json", EXPECTED / "manifest.json")
    EXPECTED_FILES = json.loads((EXPECTED / "manifest.json").read_text())
    for f in EXPECTED_FILES:
        (EXPECTED / f).parent.mkdir(parents=True, exist_ok=True)
        if not (EXPECTED / f).exists():
            fetch(f"{EXPECTED_URL}/{f}", EXPECTED / f)

os.chdir(WORK)
print("work tree:", WORK, "| expected:", EXPECTED, f"({len(EXPECTED_FILES)} files)")

In [ ]:
CHECKS = []

def check(step, name, mine, ref):
    ok = mine == ref
    CHECKS.append({"step": step, "check": name, "notebook": mine, "saved": ref, "match": bool(ok)})
    return ok

def report(step):
    rows_ = [c for c in CHECKS if c["step"] == step]
    bad = [c for c in rows_ if not c["match"]]
    print(f"[{step}] {len(rows_) - len(bad)}/{len(rows_)} checks match")
    for c in bad:
        print(f"    MISMATCH {c['check']}: notebook={c['notebook']!r}  saved={c['saved']!r}")

def check_file(step, produced, expected_name=None):
    # A file this notebook wrote, byte for byte against the expected copy
    produced = Path(produced)
    name = expected_name or produced.name
    same = produced.exists() and filecmp.cmp(produced, EXPECTED / name, shallow=False)
    check(step, f"{name} byte-identical", same, True)
    if not same and produced.exists() and produced.suffix == ".csv":
        a, b = pd.read_csv(produced), pd.read_csv(EXPECTED / name)
        print(f"    {name}: shape {a.shape} vs {b.shape}")

def plain(v):
    # numpy scalars and arrays to plain Python, as the saved JSON files were written
    if hasattr(v, "item"):
        return v.item()
    if hasattr(v, "tolist"):
        return v.tolist()
    return str(v)

def write_results(path, obj):
    text = json.dumps(obj, indent=2, sort_keys=True, default=plain, allow_nan=True)
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text if text.endswith("\n") else text + "\n")

def same_value(a, b):
    # exact equality, except that NaN equals NaN
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(same_value(a[k], b[k]) for k in a)
    if isinstance(a, list) and isinstance(b, list):
        return len(a) == len(b) and all(same_value(x, y) for x, y in zip(a, b))
    if isinstance(a, float) and isinstance(b, float) and math.isnan(a) and math.isnan(b):
        return True
    return a == b

def check_json(step, expected_name, key, mine):
    # One top-level part of an expected JSON file against the value computed here
    ref = json.loads((EXPECTED / expected_name).read_text())[key]
    got = json.loads(json.dumps(mine, sort_keys=True, default=plain, allow_nan=True))
    ok = same_value(got, ref)
    CHECKS.append({"step": step, "check": f"{expected_name}[{key}] equal", "notebook": None if ok else got,
                   "saved": None if ok else ref, "match": ok})

## Bước 1: Các hàm dùng chung

Mọi thí nghiệm của bài dùng chung các hàm dưới đây.

- `parse_dates`: đọc ngày thu thập, thử dạng `dd/mm/yyyy` trước rồi `yyyy-mm-dd`, chỉ lấy 10 ký tự đầu.
- `load`: đọc luồng, đổi nhãn thành `y` (1 là phishing), gán thời điểm `_t` cho từng dòng rồi sắp xếp theo `_t`. Với
  `spread_undated=True`, mỗi dòng không có ngày nhận một ngày giả: lấy SHA-1 của `salt + id`, chia lấy dư 10.000 để được một phân
  số trong [0, 1), rồi đặt dòng đó vào đúng vị trí phân số ấy trên khoảng từ ngày nhỏ nhất đến ngày lớn nhất. `salt` rỗng là cách
  rải của bản công bố; Bước 9d đổi `salt` để rải lại. Đặc trưng là mọi cột số trừ các cột trong `DROP` (`rank` chỉ có ở benign nên
  làm lộ nhãn; `is_https` là dấu vết của cách thu thập).
- `psi`: PSI giữa hai mẫu của một đặc trưng, chia 10 khoảng theo phân vị của mẫu tham chiếu, cộng `1e-6` vào mỗi tỷ lệ để tránh
  chia cho 0; trả 0 nếu đặc trưng có ít hơn 3 điểm cắt phân biệt.
- `fit`: ba mô hình. `rf` là rừng ngẫu nhiên 200 cây, `class_weight="balanced"`, mô hình mà mọi chính sách dùng. `cb` là CatBoost
  300 vòng, sâu 6, tốc độ học 0,1. `cblr` là stacking CatBoost với hồi quy logistic cân lớp, 5 fold, bộ học meta là hồi quy logistic.
- `run`: chạy ba chính sách (và hai kiến trúc tĩnh) trên luồng cắt thành `windows` cửa sổ bằng nhau. Ở mỗi cửa sổ từ cửa sổ 2, mỗi
  chính sách chấm F1 rồi quyết định có huấn luyện lại cho cửa sổ sau không. `periodic` huấn luyện lại khi số thứ tự cửa sổ chia hết
  cho 3. `drift` huấn luyện lại khi PSI trung bình trên mọi đặc trưng (so với cửa sổ tham chiếu của lần huấn luyện gần nhất) vượt
  0,2, hoặc F1 tụt quá 0,1 so với cửa sổ trước (F1 tham chiếu khởi đầu là 1,0). Khi huấn luyện lại, mô hình học trên toàn bộ dữ
  liệu đã thấy. Nếu truyền `trace`, hàm ghi thêm ROC-AUC, balanced accuracy, và F1 trên các dòng có vector đặc trưng chưa từng xuất
  hiện ở cửa sổ 1 (Bước 9).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score,
                             balanced_accuracy_score)

DRIFT_CSV = "data/interim/drift_compphish.csv"
OUT = "data/processed/p5"
DEDUP_OUT = "data/processed/p5/dedup"
RESULTS = "data/processed/p5/results"

# Stream settings, as stated in the paper's method section
WINDOWS, PERIOD, PSI_TAU, F1_DROP, ADWIN_DELTA = 20, 3, 0.2, 0.1, 0.002
BUDGETS = [50, 200, 800]                      # the budget table's three columns
DENSE = (25, 50, 100, 200, 400, 800, 1600)    # the dense budget grid

# rank is benign-only (a label leak), is_llm is constant, is_https is a collection artefact
DROP = {"label", "y", "is_llm", "split", "page", "id", "rank", "tier", "censored", "is_https"}


def parse_dates(col):
    s = col.astype(str).str.strip().str.slice(0, 10)
    return pd.to_datetime(s, format="%d/%m/%Y", errors="coerce").fillna(
        pd.to_datetime(s, format="%Y-%m-%d", errors="coerce"))


def dated_mask(df):
    # rows whose own event date parsed; nothing spread
    return parse_dates(df["collected_at"]).notna().to_numpy()


def psi(expected, actual, bins=10):
    qs = np.linspace(0, 1, bins + 1)
    cuts = np.unique(np.quantile(expected, qs))
    if len(cuts) < 3:
        return 0.0
    e = np.histogram(expected, bins=cuts)[0] / max(len(expected), 1) + 1e-6
    a = np.histogram(actual, bins=cuts)[0] / max(len(actual), 1) + 1e-6
    return float(np.sum((a - e) * np.log(a / e)))


def load(path, time_col="collected_at", spread_undated=False, salt=""):
    df = pd.read_csv(path, low_memory=False)
    df["y"] = (df["label"].astype(str).str.lower().isin(["phishing", "1", "spam", "smishing"])).astype(int) \
        if df["label"].dtype == object else pd.to_numeric(df["label"], errors="coerce").fillna(0).astype(int)
    if time_col in df:
        df["_t"] = parse_dates(df[time_col])
        if spread_undated and df["_t"].notna().any():
            # undated rows become a stationary background, placed by a hash of their id
            lo, hi = df["_t"].min(), df["_t"].max()
            span = (hi - lo).days or 1
            miss = df["_t"].isna()
            key = df.loc[miss, "id"].astype(str) if "id" in df else df.loc[miss].index.astype(str)
            key = salt + key  # salt "" is the released placement
            frac = key.map(lambda k: int(hashlib.sha1(k.encode()).hexdigest(), 16) % 10000 / 10000)
            df.loc[miss, "_t"] = lo + pd.to_timedelta((frac * span).round().astype(int), unit="D")
        df = df.sort_values("_t", na_position="first")
    feats = [c for c in df.columns if c not in DROP and c != time_col and c != "_t"
             and pd.api.types.is_numeric_dtype(df[c])]
    for c in feats:  # the forest rejects NaN
        if df[c].isna().any():
            df[c] = df[c].fillna(df[c].median() if df[c].notna().any() else 0)
    return df.reset_index(drop=True), feats


def fit(df, feats, kind="rf", seed=0):
    if kind == "cb":
        from catboost import CatBoostClassifier
        m = CatBoostClassifier(iterations=300, depth=6, learning_rate=0.1, random_seed=seed, verbose=False)
    elif kind == "cblr":
        from catboost import CatBoostClassifier
        from sklearn.ensemble import StackingClassifier
        from sklearn.linear_model import LogisticRegression
        m = StackingClassifier(
            estimators=[("cb", CatBoostClassifier(iterations=300, depth=6, learning_rate=0.1,
                                                  random_seed=seed, verbose=False)),
                        ("lr", LogisticRegression(max_iter=1000, class_weight="balanced"))],
            final_estimator=LogisticRegression(max_iter=1000),
            stack_method="predict_proba", cv=5, n_jobs=-1)
    else:
        m = RandomForestClassifier(n_estimators=200, class_weight="balanced", n_jobs=-1, random_state=seed)
    m.fit(df[feats], df.y)
    return m


# never-retrained architecture baselines: fitted once on window 1
STATIC_ARCH = {"static_cb": "cb", "static_cblr": "cblr"}


def windows_of(df, windows=WINDOWS):
    return [i for i in np.array_split(np.arange(len(df)), windows) if len(i)]


def run(df, feats, windows, period, psi_tau, f1_drop, include_static_arch=True, seed=0, trace=None):
    chunks = [df.iloc[i] for i in windows_of(df, windows)]
    strategies = {"static": [], "periodic": [], "drift": []}
    if include_static_arch:
        strategies.update({k: [] for k in STATIC_ARCH})
    counts = {k: 0 for k in strategies}  # retrains, the initial fit excluded

    def mean_psi(ref_df, cur_df):
        vals = [psi(ref_df[c].values, cur_df[c].values) for c in feats]
        return float(np.mean(vals)) if vals else 0.0

    base = chunks[0]
    models = {k: fit(base, feats, STATIC_ARCH.get(k, "rf"), seed=seed) for k in strategies}
    acc = base.copy()                     # everything seen so far
    ref = {k: base for k in strategies}   # PSI reference window per strategy
    last_f1 = {k: 1.0 for k in strategies}
    psi_trace = []
    if trace is not None:
        seen = set(map(tuple, base[feats].round(6).to_numpy()))
        for k in strategies:
            trace[k] = {"auc": [], "bacc": [], "novel_f1": [], "novel_share": []}

    for w in range(1, len(chunks)):
        cur = chunks[w]
        acc = pd.concat([acc, cur])
        psi_trace.append(round(mean_psi(ref["drift"], cur), 3))
        for strat in strategies:
            m = models[strat]
            if isinstance(m, RandomForestClassifier):
                m.set_params(n_jobs=1)  # one thread: tree probabilities summed in tree order, same every run
            pred = m.predict(cur[feats])
            f1 = f1_score(cur.y, pred, zero_division=0)
            strategies[strat].append(round(f1, 3))
            if trace is not None:
                sc = m.predict_proba(cur[feats])[:, 1]
                both = cur.y.nunique() == 2
                nov = ~np.array([tuple(r) in seen for r in cur[feats].round(6).to_numpy()])
                t = trace[strat]
                t["auc"].append(roc_auc_score(cur.y, sc) if both else np.nan)
                t["bacc"].append(balanced_accuracy_score(cur.y, pred) if both else np.nan)
                t["novel_f1"].append(f1_score(cur.y[nov], pred[nov], zero_division=0) if nov.any() else np.nan)
                t["novel_share"].append(float(nov[cur.y.to_numpy() == 1].mean()) if (cur.y == 1).any() else np.nan)
            # decide retraining for the next window
            retrain = False
            if strat == "periodic" and w % period == 0:
                retrain = True
            elif strat == "drift":
                if mean_psi(ref[strat], cur) > psi_tau or (last_f1[strat] - f1) > f1_drop:
                    retrain = True
            if retrain:
                models[strat] = fit(acc, feats, seed=seed); counts[strat] += 1
                ref[strat] = cur
            last_f1[strat] = f1
    return strategies, counts, psi_trace

## Bước 2: Dựng luồng

Luồng là `vn_compphish.csv` (21 đặc trưng từ vựng, cùng thứ tự dòng với `dataset_url.csv`) gắn thêm ba cột từ `dataset_url.csv`:
ngày thu thập `collected_at`, mã dòng `id` (khóa của hàm băm rải) và tầng nhãn `tier` (dùng ở Bước 10). File ghi ra là
`data/interim/drift_compphish.csv`.

Phép kiểm: SHA-256 của file luồng phải bằng giá trị đã ghi trong `p5_model_seed_metadata.json` (Bước 6 ghi lại đúng giá trị này cùng
các kết quả theo seed). Nếu hai file đầu vào hay cách ghi CSV khác đi dù một byte, phép kiểm này lệch ngay từ đây.

Bước này cũng in các con số mô tả corpus mà bài dùng: số dòng, tỷ lệ dòng có ngày, khoảng ngày của phishing có ngày.

In [ ]:
def build_input():
    u = pd.read_csv("data/processed/dataset_url.csv", low_memory=False)
    c = pd.read_csv("data/processed/vn_compphish.csv", low_memory=False)
    assert len(u) == len(c), "the two files are not row-aligned"
    c = c.copy()
    c["collected_at"] = u["collected_at"].values
    c["id"] = u["id"].values
    c["tier"] = u["tier"].values
    c.to_csv(DRIFT_CSV, index=False)


build_input()
U = pd.read_csv("data/processed/dataset_url.csv", low_memory=False)
U_DATES = parse_dates(U["collected_at"])
DATED_PCT = 100.0 * U_DATES.notna().sum() / len(U)
ph_dated = (U["label"] == "phishing") & U_DATES.notna()
print(f"rows {len(U):,}; dated {U_DATES.notna().sum():,} ({DATED_PCT:.1f}%)")
print(f"dated phishing span {U_DATES[ph_dated].min():%Y-%m} .. {U_DATES[ph_dated].max():%Y-%m}")
print(U.groupby("label").apply(lambda g: parse_dates(g["collected_at"]).notna().mean()).rename("dated share"))

ref_meta = json.loads((EXPECTED / "p5_model_seed_metadata.json").read_text())
check("2 stream", "stream file SHA-256", sha(DRIFT_CSV), ref_meta["input_sha256"])
report("2 stream")

## Bước 3: Ba chính sách trên 20 cửa sổ

Đây là thí nghiệm trung tâm của bài. Luồng (dòng không ngày được rải theo hàm băm) cắt thành 20 cửa sổ bằng nhau, khoảng 2.650 dòng
mỗi cửa sổ. Năm hàng của bảng:
- rừng ngẫu nhiên tĩnh (học một lần ở cửa sổ 1, không bao giờ cập nhật);
- CatBoost tĩnh và stacking CatBoost + hồi quy logistic tĩnh: cùng câu hỏi "không cập nhật thì mất bao nhiêu", hỏi với kiến trúc
  khác, để thấy khoảng cách không phải riêng của rừng ngẫu nhiên;
- rừng ngẫu nhiên huấn luyện lại mỗi 3 cửa sổ;
- rừng ngẫu nhiên huấn luyện lại khi PSI hoặc F1 báo trôi.

Với mỗi hàng: F1 ở cửa sổ cuối, AUTC (trung bình F1 từ cửa sổ 2 đến 20) và số lần huấn luyện lại. Kết quả mong đợi theo bài: mô
hình tĩnh kém hai chính sách cập nhật khoảng 20 điểm AUTC (0,651 so với 0,857 và 0,868). Các bước sau sẽ cho thấy khoảng cách này
đến từ đâu.

Phép kiểm: phần `overtime` của `p5_assets.json` (F1 từng cửa sổ của cả năm hàng, số lần huấn luyện lại, chuỗi PSI theo cửa sổ, số
dòng và tỷ lệ có ngày).

In [ ]:
t0 = time.time()
DF, FEATS = load(DRIFT_CSV, spread_undated=True)
print(f"n={len(DF):,} features={len(FEATS)} windows={WINDOWS}")
STRAT, COUNTS, PSI_TRACE = run(DF, FEATS, WINDOWS, PERIOD, PSI_TAU, F1_DROP)
print(f"{time.time() - t0:.0f}s")

overtime_tbl = pd.DataFrame([{"strategy": k, "end_f1": v[-1], "autc": np.mean(v), "retrains": COUNTS[k]}
                             for k, v in STRAT.items()])
display(overtime_tbl.round(3))
print("PSI per window against the drift policy's reference:", PSI_TRACE)

OVERTIME = {"rows": len(DF), "dated_pct": DATED_PCT, "window_f1": STRAT, "retrains": COUNTS,
            "psi_trace": PSI_TRACE}
check_json("3 overtime", "results/p5_assets.json", "overtime", OVERTIME)
report("3 overtime")

## Bước 4: Năm bộ giám sát trôi dữ liệu

Câu hỏi: một bộ giám sát có phát hiện được lúc dữ liệu đổi không? Dữ liệu thật không có nhãn "thời điểm trôi", nên bài dùng một nhãn
thay thế không phụ thuộc mô hình: cửa sổ `w` là "có dịch chuyển" khi, so với cửa sổ ngay trước, có ít nhất một đặc trưng mà kiểm
định KS hai mẫu vừa có ý nghĩa (ngưỡng tới hạn tiệm cận với alpha 0,01 chia Bonferroni cho 21 đặc trưng) vừa đủ lớn (`D > 0,15`).
Ngưỡng độ lớn cần thiết vì với khoảng 2.650 dòng mỗi cửa sổ, chỉ riêng ý nghĩa thống kê thì gần như cửa sổ nào cũng bị gắn cờ.

Năm bộ giám sát, cùng nhìn một mô hình rừng ngẫu nhiên học ở cửa sổ 1 và không cập nhật:
- ADWIN (delta 0,002) đọc chuỗi lỗi 0/1 của mô hình, từng dòng một, không reset; báo khi hai nửa cửa sổ trượt của nó có trung bình
  lỗi khác nhau quá cận Hoeffding. Mỗi cửa sổ chỉ đẩy lỗi vào cho đến lần báo đầu tiên trong cửa sổ đó (`any` dừng sớm);
- PSI trung bình trên 21 đặc trưng giữa hai cửa sổ liên tiếp, báo khi vượt 0,10;
- SHAP-shift: tỷ trọng mean-|SHAP| của mô hình trên 400 dòng lấy mẫu mỗi cửa sổ, báo khi khoảng cách biến phân toàn phần giữa hai
  cửa sổ liên tiếp vượt 0,05;
- ADWIN hoặc PSI;
- lịch cố định mỗi 3 cửa sổ (mốc tham chiếu "không biết gì").

Chấm theo sự kiện: mỗi cửa sổ có dịch chuyển là một sự kiện; sự kiện được phát hiện nếu bộ giám sát báo trong khoảng từ cửa sổ đó
đến 2 cửa sổ sau, và lần báo sớm nhất được tính cho nó. Tỷ lệ phát hiện = sự kiện được phát hiện / số sự kiện; tỷ lệ báo nhầm = lần
báo không được tính cho sự kiện nào / số cửa sổ không dịch chuyển; độ chính xác = lần báo được tính / tổng lần báo; độ trễ = số cửa
sổ trung bình từ sự kiện đến lần báo được tính.

Bước này còn ghi `p5_shap_trace.csv`: khoảng cách biến phân của SHAP-shift ở từng cửa sổ cạnh nhãn thay thế. Từ đó bài đọc ra rằng
ngưỡng 0,05 (chọn trên chính chuỗi này) chỉ cách giá trị cao nhất của cửa sổ không dịch chuyển rất ít.

Phép kiểm: phần `detection` của `p5_assets.json` (nhãn thay thế, chuỗi báo của từng bộ, bốn chỉ số), file `p5_shap_trace.csv`, và
phần `shap_trace` của `p5_robustness.json` (số cửa sổ có dịch chuyển, số bắt được ngay tại sự kiện, hai khoảng cách đến ngưỡng).

In [ ]:
class ADWIN:
    # Adaptive windowing over a 0/1 error stream (Bifet and Gavalda, 2007), exponential histogram
    def __init__(self, delta=0.002, max_buckets=5):
        self.delta = delta
        self.max_buckets = max_buckets
        self.buckets = []  # each: [total_sum, count]
        self.total = 0.0
        self.width = 0
        self.variance = 0.0

    def _insert_bucket(self, value):
        self.buckets.insert(0, [value, 1])
        self.width += 1
        i = 0
        size = 1
        while i < len(self.buckets):
            same = [b for b in self.buckets if b[1] == size]
            if len(same) > self.max_buckets:
                # merge the two oldest buckets of this size
                idxs = [j for j, b in enumerate(self.buckets) if b[1] == size]
                a, b = idxs[-2], idxs[-1]
                self.buckets[a][0] += self.buckets[b][0]
                self.buckets[a][1] += self.buckets[b][1]
                self.buckets.pop(b)
            size *= 2
            i += 1

    def update(self, value):
        # feed one value; True when a change was detected and the older window dropped
        self.total += value
        self._insert_bucket(value)
        return self._detect()

    def _detect(self):
        changed = False
        while True:
            cut = self._find_cut()
            if cut is None:
                break
            changed = True
        return changed

    def _find_cut(self):
        n = sum(b[1] for b in self.buckets)
        if n < 2:
            return None
        total = sum(b[0] for b in self.buckets)
        n0 = 0.0
        s0 = 0.0
        for i in range(len(self.buckets) - 1):
            n0 += self.buckets[i][1]
            s0 += self.buckets[i][0]
            n1 = n - n0
            if n1 < 1:
                break
            m0 = s0 / n0
            m1 = (total - s0) / n1
            m = 1.0 / (1.0 / n0 + 1.0 / n1)
            dd = math.log(2.0 * math.log(max(n, 2)) / self.delta)
            eps = math.sqrt(2.0 / m * 0.25 * dd) + 2.0 / 3.0 * dd / m
            if abs(m0 - m1) > eps:
                self.buckets = self.buckets[: i + 1]  # drop the older sub-window
                self.width = int(sum(b[1] for b in self.buckets))
                return True
        return None


def shap_share(model, X, n=400, seed=0):
    # normalised mean-|SHAP| attribution of the deployed model on a window subsample
    import shap
    sub = X.sample(min(n, len(X)), random_state=seed)
    sv = shap.TreeExplainer(model).shap_values(sub)
    sv = sv[1] if isinstance(sv, list) else (sv[..., 1] if getattr(sv, "ndim", 2) == 3 else sv)
    imp = np.abs(sv).mean(axis=0)
    s = imp.sum()
    return imp / s if s else imp


def ks_ground_truth(ref, cur, feats, alpha=0.01, d_min=0.15):
    # proxy shift: some feature's KS statistic is both significant (Bonferroni) and larger than d_min
    n1, n2 = len(ref), len(cur)
    if n1 < 20 or n2 < 20:
        return False, 0.0
    c_alpha = math.sqrt(-0.5 * math.log((alpha / len(feats)) / 2.0))
    crit = c_alpha * math.sqrt((n1 + n2) / (n1 * n2))
    worst = 0.0
    for c in feats:
        a = np.sort(ref[c].values)
        b = np.sort(cur[c].values)
        grid = np.union1d(a, b)
        ca = np.searchsorted(a, grid, side="right") / n1
        cb = np.searchsorted(b, grid, side="right") / n2
        worst = max(worst, float(np.max(np.abs(ca - cb))))
    return (worst > max(crit, d_min)), worst


def run_detection(df, feats, windows, period, adwin_delta, psi_det=0.10, shap_tau=0.05):
    chunks = [df.iloc[i] for i in windows_of(df, windows)]
    base = chunks[0]
    model = fit(base, feats)          # fixed deployed model
    adwin = ADWIN(delta=adwin_delta)
    for e in (model.predict(base[feats]) != base.y).astype(float):
        adwin.update(e)
    s_prev = shap_share(model, base[feats])
    truth, psi_fire, adw_fire, shp_fire, fix_fire = [], [], [], [], []
    prev = base
    for w in range(1, len(chunks)):
        cur = chunks[w]
        is_drift, _ = ks_ground_truth(prev, cur, feats)
        truth.append(is_drift)
        mean_psi = float(np.mean([psi(prev[c].values, cur[c].values) for c in feats]))
        psi_fire.append(mean_psi > psi_det)
        err = (model.predict(cur[feats]) != cur.y).astype(float)
        adw_fire.append(bool(any(adwin.update(e) for e in err)))
        s_cur = shap_share(model, cur[feats])
        shp_fire.append(0.5 * float(np.abs(s_cur - s_prev).sum()) > shap_tau)
        s_prev = s_cur
        fix_fire.append(w % period == 0)
        prev = cur
    return truth, {"ADWIN (error stream)": adw_fire,
                   "PSI (feature distribution)": psi_fire,
                   "SHAP-shift (attribution)": shp_fire,
                   "ADWIN + PSI (combined)": [a or p for a, p in zip(adw_fire, psi_fire)],
                   "Fixed-schedule (naive)": fix_fire}


def score_detector(truth, fire, horizon=2):
    onsets = [i for i, t in enumerate(truth) if t]
    detected, lags, credited = 0, [], set()
    for o in onsets:
        for j in range(o, min(len(fire), o + horizon + 1)):
            if fire[j]:
                detected += 1
                lags.append(j - o)
                credited.add(j)
                break
    alarms = [j for j, f in enumerate(fire) if f]
    fp = [j for j in alarms if j not in credited]
    neg = len(truth) - len(onsets)
    det_rate = detected / len(onsets) if onsets else float("nan")
    fa_rate = len(fp) / neg if neg else float("nan")
    prec = len(credited) / len(alarms) if alarms else float("nan")
    mean_lag = float(np.mean(lags)) if lags else float("nan")
    return det_rate, fa_rate, mean_lag, prec


t0 = time.time()
TRUTH, FIRES = run_detection(DF, FEATS, WINDOWS, PERIOD, ADWIN_DELTA)
print(f"{time.time() - t0:.0f}s; {sum(TRUTH)}/{len(TRUTH)} windows are proxy-shift windows")
SCORES = {k: dict(zip(("detection_rate", "false_alarm_rate", "mean_lag", "precision"), score_detector(TRUTH, v)))
          for k, v in FIRES.items()}
display(pd.DataFrame(SCORES).T.round(2))
DETECTION = {"truth": TRUTH, "fires": FIRES, "scores": SCORES}
check_json("4 monitors", "results/p5_assets.json", "detection", DETECTION)

Chuỗi SHAP-shift theo từng cửa sổ: cùng mô hình, cùng mẫu 400 dòng và cùng nhãn thay thế như trên, nhưng ghi giá trị khoảng cách
thay vì chỉ ghi có báo hay không.

In [ ]:
def shap_trace():
    chunks = [DF.iloc[i] for i in windows_of(DF)]
    model = fit(chunks[0], FEATS)
    prev, s_prev, rows = chunks[0], shap_share(model, chunks[0][FEATS]), []
    for w in range(1, len(chunks)):
        cur = chunks[w]
        proxy, _ = ks_ground_truth(prev, cur, FEATS)
        s_cur = shap_share(model, cur[FEATS])
        rows.append({"window": w + 1, "proxy_shift": int(proxy),
                     "tv_distance": 0.5 * float(np.abs(s_cur - s_prev).sum())})
        prev, s_prev = cur, s_cur
    out = pd.DataFrame(rows)
    out.to_csv(f"{OUT}/p5_shap_trace.csv", index=False)
    return out


SHAP_TRACE = shap_trace()
display(SHAP_TRACE.round(4))
check_file("4 monitors", f"{OUT}/p5_shap_trace.csv")

# how close the 0.05 threshold sits to the two groups of windows, read from the saved table
SHAP_TRACE = pd.read_csv(f"{OUT}/p5_shap_trace.csv")
tau = 0.05
prox, non = SHAP_TRACE[SHAP_TRACE.proxy_shift == 1], SHAP_TRACE[SHAP_TRACE.proxy_shift == 0]
fired = prox[prox.tv_distance > tau]
SHAP_NUMBERS = {"proxy_windows": len(prox), "caught_at_onset": len(fired),
                "margin_above_nonproxy": tau - non.tv_distance.max(),
                "margin_below_fired": fired.tv_distance.min() - tau}
print(SHAP_NUMBERS)
check_json("4 monitors", "results/p5_robustness.json", "shap_trace", SHAP_NUMBERS)
report("4 monitors")

## Bước 5: Biên ngân sách gán nhãn

Ở Bước 3, mọi cửa sổ đều có nhãn đầy đủ, nên chi phí duy nhất là số lần huấn luyện lại. Thực tế phải trả tiền cho từng nhãn. Ở đây
mỗi lần huấn luyện lại chỉ được mua tối đa `B` nhãn từ cửa sổ hiện tại, cộng dồn vào tập huấn luyện. Bốn cột tách hai trục thiết kế:
- `static`: không bao giờ huấn luyện lại (0 nhãn sau lần đầu);
- `periodic`: mỗi 3 cửa sổ, mua `B` dòng ngẫu nhiên;
- `drift`: khi PSI trung bình so với cửa sổ huấn luyện gần nhất vượt 0,2, mua `B` dòng ngẫu nhiên. Ở đây chỉ dùng vế PSI, vì vế F1
  cần nhãn đầy đủ của cửa sổ, thứ mà ngân sách không trả;
- `active`: cùng lịch với `periodic`, nhưng mua `B` dòng có điểm gần 0,5 nhất (bất định nhất). Thứ tự sắp xếp là `stable`: rừng 200
  cây cho rất ít giá trị xác suất khác nhau, nên có rất nhiều dòng hòa điểm; sắp xếp ổn định phá hòa theo thứ tự dòng để kết quả lặp
  lại được.

Như vậy `periodic` với `active` chỉ khác ở cách chọn dòng; `periodic` với `drift` chỉ khác ở lúc kích hoạt. Mỗi chiến lược có bộ
sinh số ngẫu nhiên riêng, để số lần huấn luyện của chiến lược này không làm đổi các dòng được rút ở chiến lược kia.

Bảng của bài dùng ba mức `B` = 50, 200, 800. Phép kiểm: phần `budget` của `p5_assets.json`, rồi cả file `p5_assets.json` từng byte
(lúc này đã đủ ba phần).

In [ ]:
def run_budget(df, feats, windows, budgets, period=3, psi_tau=0.10, seed=0):
    chunks = [df.iloc[i] for i in windows_of(df, windows)]
    base = chunks[0]
    out = {}
    for B in budgets:
        f1s = {k: [] for k in ("static", "periodic", "drift", "active")}
        labels = {k: 0 for k in f1s}
        models = {k: fit(base, feats, seed=seed) for k in f1s}
        pools = {k: base.copy() for k in f1s}
        ref = {k: base for k in f1s}
        # one RNG per strategy, so one column's draws never depend on how often another retrains
        rngs = {k: np.random.RandomState(seed) for k in f1s}
        for w in range(1, len(chunks)):
            cur = chunks[w]
            for k in f1s:
                # one thread: with hundreds of rows tied on |p - 0.5|, a last-bit difference from
                # thread order would change which rows active learning buys
                p = models[k].set_params(n_jobs=1).predict_proba(cur[feats])[:, 1]
                f1s[k].append(f1_score(cur.y, (p >= 0.5).astype(int), zero_division=0))
                if k == "static":
                    continue
                if k in ("periodic", "active"):
                    fire = (w % period == 0)
                else:  # drift: the label-free PSI term only
                    psi_mean = float(np.mean([psi(ref[k][c].values, cur[c].values) for c in feats]))
                    fire = psi_mean > psi_tau
                if not fire:
                    continue
                if k == "active":
                    # stable sort: ties (many, with a 200-tree forest) break by row order
                    take = cur.iloc[np.argsort(np.abs(p - 0.5), kind="stable")[:B]]
                else:
                    take = cur.sample(min(B, len(cur)), random_state=rngs[k])
                pools[k] = pd.concat([pools[k], take])
                labels[k] += len(take)
                models[k] = fit(pools[k], feats, seed=seed)
                ref[k] = cur
        out[B] = {k: (float(np.mean(f1s[k])), labels[k]) for k in f1s}
    return out


t0 = time.time()
BUD = run_budget(DF, FEATS, WINDOWS, BUDGETS, period=PERIOD, psi_tau=PSI_TAU)
print(f"{time.time() - t0:.0f}s")
display(pd.DataFrame({B: {k: f"{a:.3f} ({n:,})" for k, (a, n) in r.items()} for B, r in BUD.items()}).T
        .rename_axis("B (labels/retrain)"))
BUDGET = {str(b): {k: {"autc": a, "labels": n} for k, (a, n) in r.items()} for b, r in BUD.items()}
check_json("5 budget", "results/p5_assets.json", "budget", BUDGET)

write_results(f"{RESULTS}/p5_assets.json", {"overtime": OVERTIME, "detection": DETECTION, "budget": BUDGET})
check_file("5 budget", f"{RESULTS}/p5_assets.json", "results/p5_assets.json")

Lưới dày: cùng thí nghiệm ở bảy mức `B` từ 25 đến 1.600 (lưới nhân đôi, vì điều thú vị nằm ở phía ngân sách thấp), ghi
`p5_budget_frontier.csv`. Từ lưới này bài đọc ra:
- chính sách `drift` có vượt `periodic` ở mức ngân sách nào không (điểm giao: mức `B` đầu tiên mà hiệu đổi dấu từ âm sang dương);
- hiệu `drift` trừ `periodic` ở mức rẻ nhất và đắt nhất;
- `drift` mua bao nhiêu phần trăm số nhãn của `periodic`, và so ở cùng số nhãn đã mua (`drift` ở mức `B` với `periodic` ở mức mua
  đúng bấy nhiêu nhãn);
- `active` thua `periodic` ở bao nhiêu mức, và hiệu đó có thu hẹp đều khi `B` tăng không;
- điểm tốt nhất trên cả lưới và AUTC của mô hình tĩnh.

Phép kiểm: `p5_budget_frontier.csv` và `p5_budget_frontier.json` (các con số trên).

In [ ]:
GRID = tuple(sorted(set(DENSE) | set(BUDGETS)))
t0 = time.time()
dense = run_budget(DF, FEATS, WINDOWS, list(GRID), period=PERIOD, psi_tau=PSI_TAU)
print(f"{time.time() - t0:.0f}s")
FRONTIER = [{"budget": b, "policy": pol, "autc": dense[b][pol][0], "labels": dense[b][pol][1],
             "registered": int(b in BUDGETS)}
            for b in sorted(dense) for pol in ("static", "periodic", "drift", "active")]
buf = io.StringIO()
w_ = csv.DictWriter(buf, fieldnames=list(FRONTIER[0].keys()))
w_.writeheader()
w_.writerows(FRONTIER)
with open(f"{OUT}/p5_budget_frontier.csv", "w", encoding="utf-8") as f:
    f.write(buf.getvalue())
display(pd.DataFrame(FRONTIER).pivot(index="budget", columns="policy", values="autc").round(4))
check_file("5 budget", f"{OUT}/p5_budget_frontier.csv")


def frontier_numbers(rows):
    bs = sorted({r["budget"] for r in rows})
    per = {r["budget"]: r["autc"] for r in rows if r["policy"] == "periodic"}
    dri = {r["budget"]: r["autc"] for r in rows if r["policy"] == "drift"}
    act = {r["budget"]: r["autc"] for r in rows if r["policy"] == "active"}
    lab = {(r["policy"], r["budget"]): r["labels"] for r in rows}
    gaps = [(b, dri[b] - per[b]) for b in bs]
    # the first grid point where drift minus periodic turns positive
    flip = next((b for (b, g), (_, gp) in zip(gaps[1:], gaps) if g > 0 >= gp), None)
    act_gaps = [act[b] - per[b] for b in bs]
    # the trigger at B against the schedule at the budget that buys as many labels
    eq = [(b, h, dri[b] - per[h]) for b in bs for h in bs
          if lab[("drift", b)] == lab[("periodic", h)] and lab[("drift", b)] > 0]
    best = max((r for r in rows if r["policy"] != "static"), key=lambda r: r["autc"])
    return {"crossover": flip, "drift_minus_periodic_cheapest": gaps[0][1],
            "drift_minus_periodic_dearest": gaps[-1][1],
            "drift_label_ratio": lab[("drift", bs[-1])] / lab[("periodic", bs[-1])],
            "matched_pairs": [list(e) for e in eq],
            "active_below_periodic": sum(1 for b in bs if act[b] < per[b]),
            "active_minus_periodic": dict(zip(map(str, bs), act_gaps)),
            "active_monotone": all(act_gaps[i] <= act_gaps[i + 1] for i in range(len(act_gaps) - 1)),
            "static_autc": next(r["autc"] for r in rows if r["policy"] == "static"),
            "best": {"policy": best["policy"], "budget": best["budget"], "autc": best["autc"],
                     "labels": best["labels"]}}


FN = frontier_numbers(FRONTIER)
print(json.dumps(FN, indent=1, default=plain)[:1500])
write_results(f"{RESULTS}/p5_budget_frontier.json", FN)
check_file("5 budget", f"{RESULTS}/p5_budget_frontier.json", "results/p5_budget_frontier.json")
report("5 budget")

## Bước 6: Độ nhạy theo seed của mô hình

Bước 3 dùng một seed. Bước này chạy lại ba chính sách rừng ngẫu nhiên (không kèm hai kiến trúc tĩnh) với seed 0 đến 4, trên cùng một
luồng cố định. Seed đổi cả rừng ban đầu lẫn mọi lần huấn luyện lại, còn luồng thì giữ nguyên. Vì vậy độ lệch chuẩn ở đây đo độ ngẫu
nhiên của mô hình, không đo sự khác nhau giữa các lát cắt thời gian. Hiệu ghép cặp `drift` trừ `periodic` được tính trên từng seed.

Bốn file: tóm tắt theo seed và chính sách (`p5_model_seed_sensitivity.csv`), F1 từng cửa sổ (`p5_model_seed_windows.csv`), hiệu ghép
cặp (`p5_model_seed_paired.csv`), và siêu dữ liệu (`p5_model_seed_metadata.json`, trong đó có SHA-256 của luồng mà Bước 2 đã đối
chiếu).

In [ ]:
SEEDS = (0, 1, 2, 3, 4)
t0 = time.time()
windows_rows, summaries = [], []
for seed in SEEDS:
    scores, counts, _ = run(DF, FEATS, WINDOWS, PERIOD, PSI_TAU, F1_DROP, include_static_arch=False, seed=seed)
    for policy, values in scores.items():
        summaries.append(dict(seed=seed, policy=policy, autc=np.mean(values), end_f1=values[-1],
                              retrains=counts[policy]))
        windows_rows.extend(dict(seed=seed, policy=policy, window=w, f1=f1) for w, f1 in enumerate(values, 2))
    print(f"seed {seed}: " + ", ".join(f"{p}={np.mean(v):.4f}" for p, v in scores.items()), flush=True)
print(f"{time.time() - t0:.0f}s")

summary = pd.DataFrame(summaries)
summary.to_csv(f"{OUT}/p5_model_seed_sensitivity.csv", index=False)
pd.DataFrame(windows_rows).to_csv(f"{OUT}/p5_model_seed_windows.csv", index=False)
paired = summary.pivot(index="seed", columns="policy", values="autc")
delta = paired["drift"] - paired["periodic"]
pd.DataFrame({"seed": delta.index, "drift_minus_periodic_autc": delta.values}).to_csv(
    f"{OUT}/p5_model_seed_paired.csv", index=False)
metadata = dict(seeds=SEEDS, windows=WINDOWS, period=PERIOD, psi_tau=PSI_TAU,
                f1_drop=F1_DROP, n_rows=len(DF), features=FEATS,
                input_sha256=sha(DRIFT_CSV),
                scope="model randomness conditional on one fixed constructed stream",
                score_precision="per-window F1 rounded to 3 decimals by existing runner",
                paired_delta_mean=float(delta.mean()), paired_delta_sd=float(delta.std(ddof=1)))
Path(f"{OUT}/p5_model_seed_metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")

display(summary.groupby("policy", sort=False)[["autc", "end_f1", "retrains"]].agg(["mean", "std"]).round(4))
print(f"paired AUTC, drift minus periodic: {delta.mean():+.4f} +/- {delta.std(ddof=1):.4f}")
for f in ("p5_model_seed_sensitivity.csv", "p5_model_seed_windows.csv", "p5_model_seed_paired.csv",
          "p5_model_seed_metadata.json"):
    check_file("6 model seeds", f"{OUT}/{f}")
report("6 model seeds")

## Bước 7: Khoảng cách 20 điểm đến từ đâu

### 7a. Thành phần từng cửa sổ

Với mỗi cửa sổ: số dòng, tỷ lệ dòng rải (không có ngày thật), tỷ lệ phishing, và (từ cửa sổ 2) độ chính xác, độ nhạy, F1 của mô hình
tĩnh cạnh F1 của hai chính sách cập nhật. Độ nhạy của mô hình tĩnh được tách thêm theo hai loại dòng phishing: có ngày thật, và
được rải.

Điều bài đọc ra: độ chính xác của mô hình tĩnh gần như không đổi; cái thay đổi là độ nhạy. Trên dòng phishing được rải, mô hình
tĩnh bắt gần hết ở mọi cửa sổ; trên dòng phishing có ngày, nó bắt rất ít. Tương quan giữa tỷ lệ dòng rải và độ nhạy của mô hình
tĩnh cũng được tính.

### 7b. Hai định dạng URL

Vì sao? Trong nguồn cộng đồng (ChongLuaDao), dòng phishing không ngày và có ngày được ghi theo hai định dạng khác nhau: dòng không
ngày mang cả tên máy chủ đầy đủ (thường có `www.`, nên có tên miền con), dòng có ngày phần lớn chỉ là tên miền đăng ký. Bảng này đo
tỷ lệ có tên miền con và độ dài URL trung vị theo nguồn và theo loại dòng. Mô hình học trên nền dòng rải học một định dạng URL và bỏ
sót định dạng kia.

### 7c. Luồng chỉ gồm dòng có ngày

Bỏ hết dòng không ngày, cắt phần còn lại thành 20 cửa sổ bằng nhau rồi chạy lại ba chính sách. Luồng này gần như toàn phishing, nên
mọi chính sách đều sát mức F1 của bộ phân loại "đoán phishing hết" (với tỷ lệ phishing `p`, mức đó là `2p / (1 + p)`). Bài đọc ra:
luồng này quá sát trần nên không đo được độ "già" của mô hình, nó không xác nhận cũng không chặn trên khoảng cách.

Phép kiểm: năm CSV (`p5_window_composition.csv`, `p5_format_split.csv`, `p5_dated_only.csv`, `p5_dated_only_windows.csv`,
`p5_dated_only_priors.csv`) và phần `window_reading` của `p5_robustness.json`.

In [ ]:
def window_composition():
    dated = dated_mask(DF)
    idx = windows_of(DF)
    strat, _, _ = run(DF, FEATS, WINDOWS, PERIOD, PSI_TAU, F1_DROP, include_static_arch=False)
    static = fit(DF.iloc[idx[0]], FEATS)
    rows = []
    for w, i in enumerate(idx):
        cur = DF.iloc[i]
        r = {"window": w + 1, "rows": len(cur), "spread_share": float(1 - dated[i].mean()),
             "prior": float(cur.y.mean())}
        if w > 0:
            pred = static.predict(cur[FEATS])
            pos = cur.y.to_numpy() == 1
            dm = dated[i]
            for tag, m in (("dated", pos & dm), ("spread", pos & ~dm)):
                r[f"static_recall_{tag}"] = float(pred[m].mean()) if m.any() else float("nan")
                r[f"positives_{tag}"] = int(m.sum())
            r.update({"static_precision": precision_score(cur.y, pred, zero_division=0),
                      "static_recall": recall_score(cur.y, pred, zero_division=0),
                      "static_f1": f1_score(cur.y, pred, zero_division=0),
                      "periodic_f1": strat["periodic"][w - 1], "drift_f1": strat["drift"][w - 1]})
        rows.append(r)
    out = pd.DataFrame(rows)
    out.to_csv(f"{OUT}/p5_window_composition.csv", index=False)
    return out


t0 = time.time()
WIN = window_composition()
print(f"{time.time() - t0:.0f}s")
display(WIN[["window", "rows", "spread_share", "prior", "static_precision", "static_recall",
             "static_recall_dated", "positives_dated", "static_recall_spread", "periodic_f1", "drift_f1"]].round(3))
check_file("7 composition", f"{OUT}/p5_window_composition.csv")

In [ ]:
def format_split():
    src = U[["id", "source"]]
    df = DF.merge(src, on="id", how="left")
    dated = dated_mask(df)
    rows = []
    for source, g in df[df.y == 1].groupby("source"):
        for tag, m in (("dated", dated[g.index]), ("undated", ~dated[g.index])):
            x = g[m]
            if len(x):
                rows.append({"source": source, "rows_kind": tag, "rows": len(x),
                             "subdomain_share": float((x.subdom_cnt > 0).mean()),
                             "median_url_len": float(x.url_len.median())})
    out = pd.DataFrame(rows)
    out.to_csv(f"{OUT}/p5_format_split.csv", index=False)
    return out


FMT = format_split()
display(FMT.round(3))
check_file("7 composition", f"{OUT}/p5_format_split.csv")

In [ ]:
def dated_only():
    df, feats = load(DRIFT_CSV, spread_undated=False)
    df = df[df["_t"].notna()].reset_index(drop=True)
    strat, counts, _ = run(df, feats, WINDOWS, PERIOD, PSI_TAU, F1_DROP, include_static_arch=False)
    out = pd.DataFrame([{"stream": "dated_only", "rows": len(df), "prior": float(df.y.mean()), "policy": k,
                         "autc": float(np.mean(v)), "end_f1": float(v[-1]), "retrains": counts[k]}
                        for k, v in strat.items()])
    out.to_csv(f"{OUT}/p5_dated_only.csv", index=False)
    win = pd.DataFrame({k: v for k, v in strat.items()})
    win.insert(0, "window", range(2, 2 + len(win)))
    win.to_csv(f"{OUT}/p5_dated_only_windows.csv", index=False)
    idx = windows_of(df)
    pd.DataFrame({"window": range(1, len(idx) + 1),
                  "prior": [float(df.y.iloc[i].mean()) for i in idx]}).to_csv(
        f"{OUT}/p5_dated_only_priors.csv", index=False)
    return out


D_ONLY = dated_only()
display(D_ONLY.round(3))
for f in ("p5_dated_only.csv", "p5_dated_only_windows.csv", "p5_dated_only_priors.csv"):
    check_file("7 composition", f"{OUT}/{f}")

Các con số bài đọc từ ba bảng trên: tương quan giữa tỷ lệ dòng rải và độ nhạy của mô hình tĩnh; trên luồng chỉ-dòng-có-ngày, mức
F1 "đoán phishing hết" trung bình, số cửa sổ mà mỗi chính sách thấp hơn mức đó, và phần khoảng cách đến riêng từ cửa sổ cuối.

In [ ]:
def floor(p):
    # F1 of the classifier that answers phishing for every row, at positive rate p
    return 2 * p / (1 + p)


sc = WIN.dropna(subset=["static_f1"])
rho = float(np.corrcoef(sc.spread_share, sc.static_recall)[0, 1])
dw = pd.read_csv(f"{OUT}/p5_dated_only_windows.csv")
dprior = pd.read_csv(f"{OUT}/p5_dated_only_priors.csv")["prior"]
fl = floor(dprior.iloc[1:].to_numpy())
below = {k: int((dw[k].to_numpy() < fl).sum()) for k in ("static", "periodic", "drift")}
gap_w = (dw[["periodic", "drift"]].max(axis=1) - dw["static"]).to_numpy()
last_share = float(gap_w[-1] / gap_w.sum()) if gap_w.sum() > 0 else float("nan")
WINDOW_READING = {"spread_recall_corr": rho, "dated_floor_mean": float(fl.mean()),
                  "below_floor": below, "last_window_gap_share": last_share}
print(WINDOW_READING)
rd = sc[sc.positives_dated >= 300]
print(f"static recall on spread positives {sc.static_recall_spread.min():.3f}-{sc.static_recall_spread.max():.3f}; "
      f"on dated positives {rd.static_recall_dated.min():.3f}-{rd.static_recall_dated.max():.3f} "
      f"(windows with at least 300 dated positives)")
check_json("7 composition", "results/p5_robustness.json", "window_reading", WINDOW_READING)
report("7 composition")

## Bước 8: Lưới ngân sách lặp trên năm seed

Lưới dày của Bước 5 chỉ dùng seed 0. Bước này lặp lại cả lưới với seed 0 đến 4; seed đặt cùng lúc rừng ngẫu nhiên và các lần rút
nhãn ngẫu nhiên. Đây là bước lâu nhất của notebook.

Từ bảng này bài đọc: `active` có thua `periodic` ở mọi mức và mọi seed không; hiệu `drift` trừ `periodic` (trung bình và độ lệch
chuẩn theo seed) ở từng mức `B`; ở mức rẻ nhất, mức thứ hai và mức đắt nhất có bao nhiêu seed mà `drift` dẫn; `drift` mua bao nhiêu
phần trăm số nhãn của `periodic`; và so ở cùng số nhãn đã mua thì `drift` thua ở bao nhiêu cặp seed–ngân sách.

Phép kiểm: `p5_budget_seeds.csv`, phần `budget_seeds` của `p5_robustness.json`, rồi cả file `p5_robustness.json` từng byte.

In [ ]:
t0 = time.time()
rows = []
for s in range(5):
    res = run_budget(DF, FEATS, WINDOWS, list(GRID), period=PERIOD, psi_tau=PSI_TAU, seed=s)
    for B, d in res.items():
        for k, (autc, labels) in d.items():
            rows.append({"seed": s, "budget": B, "policy": k, "autc": autc, "labels": labels})
    print(f"[seed {s}] done, {time.time() - t0:.0f}s", flush=True)
BSEEDS = pd.DataFrame(rows)
BSEEDS.to_csv(f"{OUT}/p5_budget_seeds.csv", index=False)
check_file("8 budget seeds", f"{OUT}/p5_budget_seeds.csv")

w_ = BSEEDS.pivot_table(index=["seed", "budget"], columns="policy", values="autc")
gaps_tbl = pd.DataFrame({"drift_minus_periodic": w_["drift"] - w_["periodic"],
                         "active_minus_periodic": w_["active"] - w_["periodic"]})
display(gaps_tbl.groupby("budget").agg(["mean", "std"]).round(4))

In [ ]:
def label_note(s):
    # which labels each policy buys, and the trigger against the schedule at equal spend
    lab = s.pivot_table(index=["seed", "budget"], columns="policy", values="labels")
    auc = s.pivot_table(index=["seed", "budget"], columns="policy", values="autc")
    if (lab.drift == lab.periodic).all():
        return {}
    pairs = []
    for (sd, b), r in lab.iterrows():
        for (sd2, h), r2 in lab.xs(sd, level="seed", drop_level=False).iterrows():
            if r.drift == r2.periodic and r.drift > 0:
                pairs.append((b, h, auc.loc[(sd, b), "drift"] - auc.loc[(sd2, h), "periodic"]))
    p = pd.DataFrame(pairs, columns=["b", "h", "d"])
    g = p.groupby("b").d
    return {"drift_label_share": float((lab.drift / lab.periodic).median()), "matched_pairs": len(p),
            "matched_pairs_drift_behind": int((p.d < 0).sum()),
            "matched_gap_mean_by_budget": {str(k): v for k, v in g.mean().items()}}


def budget_seed_numbers(s):
    w = s.pivot_table(index=["seed", "budget"], columns="policy", values="autc")
    dp = (w["drift"] - w["periodic"]).groupby("budget")
    ap = (w["active"] - w["periodic"]).groupby("budget")
    bs = sorted(s.budget.unique())
    lo, second, hi = bs[0], bs[1], bs[-1]
    wins = lambda b: int(((w["drift"] - w["periodic"]).xs(b, level="budget") > 0).sum())
    out = {"seeds": s.seed.nunique(), "active_below_everywhere": bool(((w["active"] - w["periodic"]) < 0).all()),
           "active_worst_budget": int(ap.mean().idxmin()),
           "drift_minus_periodic": {str(b): [dp.mean()[b], dp.std()[b]] for b in bs},
           "active_minus_periodic": {str(b): [ap.mean()[b], ap.std()[b]] for b in bs},
           "drift_ahead_seeds": {str(lo): wins(lo), str(second): wins(second), str(hi): wins(hi)}}
    out.update(label_note(s))
    return out


BUDGET_SEEDS = budget_seed_numbers(BSEEDS)
print(json.dumps(BUDGET_SEEDS, indent=1, default=plain))
check_json("8 budget seeds", "results/p5_robustness.json", "budget_seeds", BUDGET_SEEDS)

write_results(f"{RESULTS}/p5_robustness.json", {"window_reading": WINDOW_READING, "shap_trace": SHAP_NUMBERS,
                                                "budget_seeds": BUDGET_SEEDS})
check_file("8 budget seeds", f"{RESULTS}/p5_robustness.json", "results/p5_robustness.json")
report("8 budget seeds")

## Bước 9: Bốn cách dựng luồng từ cùng một corpus

Giữ nguyên corpus và các chính sách, chỉ đổi cách dựng luồng:
- `released`: các dòng đã công bố, dòng không ngày rải theo hàm băm (luồng của Bước 3);
- `nowww`: cùng các dòng đó, bỏ `www.` ở đầu mọi URL rồi tính lại 21 đặc trưng; số dòng, ngày và `id` không đổi;
- `dedup`: mỗi URL một dòng (bỏ scheme và `www.` ở đầu để so; ưu tiên dòng có ngày, rồi dòng không có `www.`, rồi dòng đứng trước),
  dòng không ngày còn lại vẫn rải theo hàm băm;
- `dated`: chỉ dòng có ngày (luồng của Bước 7c), không có gì được rải.

### 9a. Các con số của corpus và hai luồng mới

Trước hết là các con số mà phần dựng luồng của bài dựa vào: số dòng có ngày, số dòng phishing không ngày, bao nhiêu trong đó là bản
có `www.` của một URL cũng xuất hiện ở dòng có ngày, và khử trùng lặp bỏ đi bao nhiêu dòng. Sau đó dựng hai file luồng mới.

Luồng `nowww` cần tính lại đặc trưng, nên ô thứ hai dưới đây chứa bộ trích 25 cột theo lược đồ CompPhish (21 cột được dùng làm đặc
trưng; `url`, `dom`, `tld` là chuỗi, `is_https` bị loại). Mọi đặc trưng đếm trên URL đã bỏ scheme; tên miền đăng ký và đuôi tên miền
lấy theo Public Suffix List đóng gói sẵn trong `tldextract` (không tải từ mạng).

Lưu ý: file đặc trưng đã công bố được tính bằng một bản Public Suffix List cũ hơn. Bản đi kèm `tldextract` 5.3.1 coi một số đuôi
tỉnh như `binhphuoc.vn`, `binhdinh.vn` là đuôi công cộng, nên vài chục dòng có `dom`, `tld`, `subdom_cnt` khác với file đã công bố.
Ô dưới in số dòng đó. Luồng `nowww` được tính bằng bộ trích hiện tại, nên ở những dòng này nó khác luồng `released` thêm một chút
ngoài việc bỏ `www.`.

In [ ]:
def bare_host(urls):
    return (urls.astype(str).str.lower().str.replace(r"^[a-z][a-z0-9+.-]*://", "", regex=True)
            .str.replace(r"^www\.", "", regex=True))


def construction_numbers():
    dated = U_DATES.notna()
    host = U.url_norm.astype(str).str.lower().str.replace(r"^[a-z][a-z0-9+.-]*://", "", regex=True)
    bare = host.str.replace(r"^www\.", "", regex=True)
    www = host.str.startswith("www.")
    ph = U.label.astype(str).str.lower() == "phishing"
    und = (~dated) & ph
    copy = und & bare.isin(set(bare[dated])) & www
    return {"rows": len(U), "dated": int(dated.sum()), "undated_phishing": int(und.sum()),
            "www_copies": int(copy.sum()), "removed_by_dedup": int(len(U) - bare.nunique())}


CONSTRUCTION = construction_numbers()
print(CONSTRUCTION, f"-> {100 * CONSTRUCTION['www_copies'] / CONSTRUCTION['undated_phishing']:.1f}% of undated phishing")
check_json("9a construction", "results/p5_sweep.json", "construction", CONSTRUCTION)

# one row per URL: the dated row first, then the row without www., then row order
c_all = pd.read_csv("data/processed/vn_compphish.csv", low_memory=False)
host = U["url_norm"].astype(str).str.lower().str.replace(r"^[a-z][a-z0-9+.-]*://", "", regex=True)
k = pd.DataFrame({"i": np.arange(len(U)), "bare": bare_host(U["url_norm"]),
                  "dated": U_DATES.notna(), "www": host.str.startswith("www.")})
k = k.sort_values(["bare", "dated", "www", "i"], ascending=[True, False, True, True])
keep = np.sort(k.drop_duplicates("bare")["i"].to_numpy())
ud, cd = U.iloc[keep].reset_index(drop=True), c_all.iloc[keep].reset_index(drop=True)
print(f"kept {len(ud):,} of the corpus's {len(U):,} rows, one per URL")

DEDUP_STREAM = f"{DEDUP_OUT}/drift_compphish_dedup.csv"
cd = cd.copy()
for col in ("collected_at", "id", "tier"):
    cd[col] = ud[col].values
cd.to_csv(DEDUP_STREAM, index=False)
check("9a construction", "one-row-per-URL rows", len(cd), CONSTRUCTION["rows"] - CONSTRUCTION["removed_by_dedup"])

In [ ]:
import tldextract

_EXT = tldextract.TLDExtract(suffix_list_urls=())  # the snapshot bundled with the library, no network
COLUMNS = ["url", "url_len", "dom", "dom_len", "is_ip", "tld", "tld_len", "subdom_cnt",
           "letter_cnt", "digit_cnt", "special_cnt", "eq_cnt", "qm_cnt", "amp_cnt",
           "dot_cnt", "dash_cnt", "under_cnt", "letter_ratio", "digit_ratio", "spec_ratio",
           "is_https", "slash_cnt", "entropy", "path_len", "query_len"]
_IP_RE = re.compile(r"^\d{1,3}(\.\d{1,3}){3}$")


def _entropy(s):
    if not s:
        return 0.0
    n = len(s)
    return -sum((c / n) * log2(c / n) for c in Counter(s).values())


def extract(url):
    url = str(url).strip()
    p = urlparse(url if re.match(r"^[a-z]+://", url, re.I) else "http://" + url)
    host = (p.hostname or "").lower()
    core = re.sub(r"^[a-z]+://", "", url, flags=re.I)  # every lexical count is scheme-independent
    ext = _EXT(url if re.match(r"^[a-z]+://", url, re.I) else "http://" + url)
    suffix = ext.suffix
    dom = ext.registered_domain or host
    subdom = ext.subdomain
    url_len = len(core)
    letter_cnt = sum(c.isalpha() for c in core)
    digit_cnt = sum(c.isdigit() for c in core)
    special_cnt = url_len - letter_cnt - digit_cnt
    return {"url": url, "url_len": url_len, "dom": dom, "dom_len": len(dom),
            "is_ip": 1 if _IP_RE.match(host) else 0, "tld": suffix, "tld_len": len(suffix),
            "subdom_cnt": len([s for s in subdom.split(".") if s]) if subdom else 0,
            "letter_cnt": letter_cnt, "digit_cnt": digit_cnt, "special_cnt": special_cnt,
            "eq_cnt": core.count("="), "qm_cnt": core.count("?"), "amp_cnt": core.count("&"),
            "dot_cnt": core.count("."), "dash_cnt": core.count("-"), "under_cnt": core.count("_"),
            "letter_ratio": letter_cnt / url_len if url_len else 0.0,
            "digit_ratio": digit_cnt / url_len if url_len else 0.0,
            "spec_ratio": special_cnt / url_len if url_len else 0.0,
            "is_https": 1 if url.lower().startswith("https") else 0,
            "slash_cnt": core.count("/"), "entropy": _entropy(core),
            "path_len": len(p.path.strip("/")), "query_len": len(p.query)}


def to_label(v):
    s = str(v).strip().lower()
    if s in ("1", "phishing", "phish", "spam", "smishing", "malicious"):
        return 1
    if s in ("0", "benign", "legit", "legitimate", "ham"):
        return 0
    return ""


def align(urls_df):
    # the URL table written to CSV and read back as text, featurised row by row, written and read again;
    # columns: the 25 features, label, then split and tier carried over from the input
    rows = list(csv.DictReader(io.StringIO(urls_df.to_csv(index=False))))
    buf = io.StringIO()
    wr = csv.DictWriter(buf, fieldnames=COLUMNS + ["label", "split", "tier"])
    wr.writeheader()
    for r in rows:
        url = (r.get("url_norm") or "").strip()
        if not url:
            continue
        feat = extract(url)
        feat["label"], feat["split"], feat["tier"] = to_label(r.get("label", "")), r.get("split", ""), r.get("tier", "")
        wr.writerow(feat)
    return pd.read_csv(io.StringIO(buf.getvalue()), low_memory=False)


# the released feature file was computed with an older Public Suffix List snapshot: a few provincial
# .vn suffixes (binhphuoc.vn, binhdinh.vn, ...) are public suffixes in the one bundled now
recomputed = align(U)
diff = [c for c in ("dom", "dom_len", "tld", "tld_len", "subdom_cnt")
        if not recomputed[c].astype(str).equals(c_all[c].astype(str))]
n_diff = int((recomputed[diff].astype(str) != c_all[diff].astype(str)).any(axis=1).sum())
print(f"rows whose suffix-derived features differ from the released file: {n_diff} ({diff})")

# www. removed from every URL, features recomputed; rows, dates and ids unchanged
WWW = r"^((?:[A-Za-z][A-Za-z0-9+.-]*://)?)www\."
un = U.copy()
for col in ("url", "url_norm"):
    un[col] = un[col].astype(str).str.replace(WWW, r"\1", regex=True, case=False)
cn = align(un)
assert len(cn) == len(un)
for col in ("collected_at", "id", "tier"):
    cn[col] = un[col].values
NOWWW_STREAM = f"{DEDUP_OUT}/drift_compphish_nowww.csv"
cn.to_csv(NOWWW_STREAM, index=False)
print("URLs still starting with www.:", int(un["url_norm"].str.contains(r"^(?:[a-z][a-z0-9+.-]*://)?www\.", case=False).sum()))
report("9a construction")

### 9b. Ba chính sách trên luồng một-dòng-mỗi-URL

Chạy lại Bước 3 (cả năm hàng) và Bước 7a (thành phần cửa sổ, không tách theo loại dòng) trên luồng `dedup`. Phép kiểm: ba CSV
`p5_dedup_overtime.csv` (F1 từng cửa sổ và PSI), `p5_dedup_summary.csv` (AUTC, F1 cuối, số lần huấn luyện lại),
`p5_dedup_windows.csv`.

In [ ]:
t0 = time.time()
dfd, featsd = load(DEDUP_STREAM, spread_undated=True)
strat_d, counts_d, psi_d = run(dfd, featsd, WINDOWS, PERIOD, PSI_TAU, F1_DROP)
win = pd.DataFrame(strat_d)
win.insert(0, "window", range(2, 2 + len(win)))
win["psi"] = psi_d
win.to_csv(f"{DEDUP_OUT}/p5_dedup_overtime.csv", index=False)
summ = pd.DataFrame([{"policy": k, "autc": float(np.mean(v)), "end_f1": float(v[-1]), "retrains": counts_d[k]}
                     for k, v in strat_d.items()])
summ.insert(0, "rows", len(dfd))
summ.to_csv(f"{DEDUP_OUT}/p5_dedup_summary.csv", index=False)
display(summ.round(3))

dated = dated_mask(dfd)
idx = windows_of(dfd)
static = fit(dfd.iloc[idx[0]], featsd)
rows = []
for w, i in enumerate(idx):
    cur = dfd.iloc[i]
    r = {"window": w + 1, "rows": len(cur), "spread_share": float(1 - dated[i].mean()), "prior": float(cur.y.mean())}
    if w > 0:
        pred = static.predict(cur[featsd])
        r.update({"static_precision": precision_score(cur.y, pred, zero_division=0),
                  "static_recall": recall_score(cur.y, pred, zero_division=0),
                  "static_f1": f1_score(cur.y, pred, zero_division=0)})
    rows.append(r)
pd.DataFrame(rows).to_csv(f"{DEDUP_OUT}/p5_dedup_windows.csv", index=False)
print(f"{time.time() - t0:.0f}s")
for f in ("p5_dedup_overtime.csv", "p5_dedup_summary.csv", "p5_dedup_windows.csv"):
    check_file("9b dedup stream", f"{DEDUP_OUT}/{f}", f"dedup/{f}")
report("9b dedup stream")

### 9c. Bốn cách dựng, đo bốn kiểu

Với mỗi cách dựng và mỗi chính sách rừng ngẫu nhiên: AUTC trên F1 (như Bước 3), AUTC trên ROC-AUC và trên balanced accuracy (hai
thước đo không phụ thuộc tỷ lệ phishing của cửa sổ), và AUTC trên F1 chỉ tính các dòng có vector 21 đặc trưng chưa từng xuất hiện
ở cửa sổ 1 (dòng "chưa thấy", loại trừ khả năng mô hình tĩnh chỉ nhớ lại vector đã học). Kèm theo: tỷ lệ phishing chưa thấy, tỷ lệ
phishing của cửa sổ 1, tỷ lệ dòng rải ở cửa sổ 1, khoảng tỷ lệ phishing qua các cửa sổ. Thêm chuỗi PSI (so với cửa sổ tham chiếu
của chính sách `drift`) của từng cách dựng.

Phép kiểm: `p5_construction_sweep.csv` và `p5_psi_traces.csv`.

In [ ]:
SALTS = [""] + [f"s{k}:" for k in range(1, 10)]  # "" is the released placement
CONS = (("released", DRIFT_CSV), ("nowww", NOWWW_STREAM), ("dedup", DEDUP_STREAM), ("dated", DRIFT_CSV))


def one_construction(name, path, salt):
    df, feats = load(path, spread_undated=(name != "dated"), salt=salt)
    if name == "dated":
        df = df[df["_t"].notna()].reset_index(drop=True)
    tr = {}
    strat, counts, psi_trace = run(df, feats, WINDOWS, PERIOD, PSI_TAU, F1_DROP, include_static_arch=False, trace=tr)
    idx = windows_of(df)
    pri = [float(df.y.iloc[i].mean()) for i in idx]
    dated = dated_mask(df)
    rows = []
    for k, v in strat.items():
        t = tr[k]
        rows.append({"construction": name, "salt": salt, "rows": len(df), "policy": k,
                     "autc": float(np.mean(v)), "end_f1": float(v[-1]), "retrains": counts[k],
                     "auc_autc": float(np.nanmean(t["auc"])), "bacc_autc": float(np.nanmean(t["bacc"])),
                     "novel_autc": float(np.nanmean(t["novel_f1"])),
                     "novel_pos_share": float(np.nanmean(t["novel_share"])),
                     "w1_prior": pri[0], "w1_spread": float(1 - dated[idx[0]].mean()),
                     "prior_min": min(pri), "prior_max": max(pri)})
    return rows, psi_trace


t0 = time.time()
sweep_rows, traces = [], []
for name, path in CONS:
    r, psi_trace = one_construction(name, path, "")
    sweep_rows += r
    traces.append(pd.DataFrame({"construction": name, "window": range(2, 2 + len(psi_trace)),
                                "psi_vs_reference": psi_trace}))
    print(f"[sweep] {name} done, {time.time() - t0:.0f}s", flush=True)
SWEEP = pd.DataFrame(sweep_rows)
SWEEP.to_csv(f"{DEDUP_OUT}/p5_construction_sweep.csv", index=False)
pd.concat(traces).to_csv(f"{DEDUP_OUT}/p5_psi_traces.csv", index=False)
display(SWEEP.drop(columns="salt").round(3))
for f in ("p5_construction_sweep.csv", "p5_psi_traces.csv"):
    check_file("9c sweep", f"{DEDUP_OUT}/{f}", f"dedup/{f}")
report("9c sweep")

### 9d. Rải lại theo chín muối khác

Cách rải theo hàm băm là một lựa chọn cụ thể. Bước này rải lại ba luồng có dòng được rải (`released`, `nowww`, `dedup`) với chín
chuỗi muối khác (`s1:` đến `s9:` đặt trước `id` khi băm), cộng với cách rải gốc là mười cách. Câu hỏi: khoảng cách giữa các cách dựng
có lớn hơn khoảng mà riêng việc rải lại làm dịch chuyển mỗi cách dựng không?

Phép kiểm: `p5_construction_salts.csv`.

In [ ]:
t0 = time.time()
salt_rows = [r for r in sweep_rows if r["construction"] != "dated"]
for salt in SALTS[1:]:
    for name, path in CONS[:3]:
        salt_rows += one_construction(name, path, salt)[0]
    print(f"[salt {salt}] done, {time.time() - t0:.0f}s", flush=True)
pd.DataFrame(salt_rows).to_csv(f"{DEDUP_OUT}/p5_construction_salts.csv", index=False)
check_file("9d salts", f"{DEDUP_OUT}/p5_construction_salts.csv", "dedup/p5_construction_salts.csv")
report("9d salts")

### 9e. Đọc kết quả

Khoảng cách ở mỗi cách dựng là AUTC của chính sách cập nhật tốt hơn trừ AUTC của mô hình tĩnh, tính trên F1, ROC-AUC, balanced
accuracy và F1 dòng chưa thấy. Với chín muối: khoảng nhỏ nhất–lớn nhất của khoảng cách F1 và ROC-AUC ở từng cách dựng.

Thêm hai con số:
- tỷ lệ phishing ở cửa sổ 2–20 của luồng `released` có URL (đã bỏ scheme và `www.`) trùng một URL ở cửa sổ 1. So với tỷ lệ trùng
  vector đặc trưng ở 9c, con số này cho thấy 21 đặc trưng từ vựng thô đến mức nào: rất nhiều dòng trùng vector mà không trùng URL;
- PSI "mù" với cái tạo ra khoảng cách: trong các cửa sổ mà mô hình tĩnh bắt không quá 25% phishing có ngày (cửa sổ có ít nhất 300
  dòng phishing có ngày), PSI so với cửa sổ tham chiếu của chính sách `drift` nằm trong khoảng nào, cho đến trước lần đầu nó vượt 0,2.

Phép kiểm: ba phần `salt_ranges`, `url_twin_share`, `psi_blind` của `p5_sweep.json`, rồi cả file từng byte.

In [ ]:
def gaps(x, col):
    # better updating policy minus the static model, on the AUTC of metric col
    return max(x.loc["periodic", col], x.loc["drift", col]) - x.loc["static", col]


ORDER = ("released", "nowww", "dedup", "dated")
g = {c: x.set_index("policy") for c, x in SWEEP.groupby("construction")}
display(pd.DataFrame({c: {"rows": int(g[c]["rows"].iloc[0]), "gap F1": gaps(g[c], "autc"),
                          "gap ROC-AUC": gaps(g[c], "auc_autc"), "gap bal. acc.": gaps(g[c], "bacc_autc"),
                          "unseen share": g[c]["novel_pos_share"].iloc[0], "gap unseen F1": gaps(g[c], "novel_autc"),
                          "w1 prior": g[c]["w1_prior"].iloc[0]}
                      for c in ORDER}).T.round(3))

sal = pd.read_csv(f"{DEDUP_OUT}/p5_construction_salts.csv")
srange = {}
for c, x in sal.groupby("construction"):
    per = [y.set_index("policy") for _, y in x.groupby(x["salt"].fillna(""))]
    gs = [gaps(y, "autc") for y in per]
    ga = [gaps(y, "auc_autc") for y in per]
    srange[c] = (min(gs), max(gs), len(gs), min(ga), max(ga))
SALT_RANGES = {c: dict(zip(("f1_min", "f1_max", "salts", "auc_min", "auc_max"), v)) for c, v in srange.items()}
display(pd.DataFrame(SALT_RANGES).T.round(3))
check_json("9e reading", "results/p5_sweep.json", "salt_ranges", SALT_RANGES)


def url_twin_share():
    b = DF.id.astype(str).map(dict(zip(U.id.astype(str), bare_host(U.url_norm))))
    idx = windows_of(DF)
    w1 = set(b.iloc[idx[0]])
    later = np.concatenate(idx[1:])
    ph = DF.y.iloc[later].to_numpy() == 1
    return float(b.iloc[later][ph].isin(w1).mean())


TWIN = url_twin_share()
print(f"later-window phishing repeating a window-1 feature vector: {1 - g['released']['novel_pos_share'].iloc[0]:.3f}; "
      f"repeating a window-1 URL: {TWIN:.4f}")
check_json("9e reading", "results/p5_sweep.json", "url_twin_share", TWIN)

tr = pd.read_csv(f"{DEDUP_OUT}/p5_psi_traces.csv")
tr = tr[tr.construction == "released"].set_index("window").psi_vs_reference
win_c = pd.read_csv(f"{OUT}/p5_window_composition.csv").set_index("window")
low = win_c[(win_c.positives_dated >= 300) & (win_c.static_recall_dated <= 0.25)].index
first_fire = int(tr[tr > 0.2].index.min()) if (tr > 0.2).any() else None
w0 = int(min(low))
w1 = (first_fire - 1) if first_fire else int(max(low))
quiet = tr.loc[w0:w1]
PSI_BLIND = {"first_window": w0, "last_window": w1, "psi_min": float(quiet.min()), "psi_max": float(quiet.max()),
             "first_fire": first_fire,
             "static_recall_dated_max": float(win_c.loc[w0:w1, "static_recall_dated"].max())}
print(PSI_BLIND)
check_json("9e reading", "results/p5_sweep.json", "psi_blind", PSI_BLIND)

write_results(f"{RESULTS}/p5_sweep.json", {"construction": CONSTRUCTION, "salt_ranges": SALT_RANGES,
                                           "url_twin_share": TWIN, "psi_blind": PSI_BLIND})
check_file("9e reading", f"{RESULTS}/p5_sweep.json", "results/p5_sweep.json")
report("9e reading")

## Bước 10: Độ nhạy với nhãn sai

Kiểm toán corpus ước tính 12,1% tin "phishing" lấy mẫu thực ra là trang hợp lệ, phần lớn ở tầng nguồn `bronze`. Kiểm toán cho một
tỷ lệ chứ không chỉ ra từng dòng sai, nên đây là phép thử chịu đựng, không phải khôi phục nhãn thật: trong mỗi lần trong 20 lần
chạy (seed 0 đến 19), chọn ngẫu nhiên 12,1% số dòng phishing tầng `bronze` của luồng, đổi nhãn thành benign, rồi chạy lại ba chính
sách rừng ngẫu nhiên từ đầu. Ghi AUTC gốc, AUTC sau khi đổi nhãn, hiệu, và số lần huấn luyện lại.

Phép kiểm: `p5_label_noise_sensitivity.csv`.

In [ ]:
RATE = 0.121
t0 = time.time()
base_scores, _, _ = run(DF, FEATS, WINDOWS, PERIOD, PSI_TAU, F1_DROP, include_static_arch=False)
baseline = {k: float(np.mean(v)) for k, v in base_scores.items()}
eligible = DF.index[(DF.y == 1) & DF.tier.astype(str).str.lower().eq("bronze")].to_numpy()
n_flip = int(round(RATE * len(eligible)))
records = []
for seed in range(20):
    perturbed = DF.copy()
    chosen = np.random.RandomState(seed).choice(eligible, n_flip, replace=False)
    perturbed.loc[chosen, "y"] = 0
    result, counts, _ = run(perturbed, FEATS, WINDOWS, PERIOD, PSI_TAU, F1_DROP, include_static_arch=False)
    for policy, values in result.items():
        autc = float(np.mean(values))
        records.append({"seed": seed, "policy": policy, "noise_rate": RATE,
                        "n_bronze_positive": len(eligible), "n_relabelled": n_flip,
                        "baseline_autc": baseline[policy], "corrected_autc": autc,
                        "delta_autc": autc - baseline[policy], "retrains": counts[policy]})
    print(f"replicate {seed + 1}/20, {time.time() - t0:.0f}s", flush=True)
NOISE = pd.DataFrame(records)
NOISE.to_csv(f"{OUT}/p5_label_noise_sensitivity.csv", index=False, float_format="%.6f")
print(f"{len(eligible):,} bronze phishing rows, {n_flip:,} relabelled per replicate")
display(NOISE.groupby("policy", sort=False)[["baseline_autc", "corrected_autc", "delta_autc", "retrains"]]
        .agg(["mean", "std"]).round(3))
check_file("10 label noise", f"{OUT}/p5_label_noise_sensitivity.csv")
report("10 label noise")

## Bước 11: Một corpus có ngày ở mọi dòng

PhishVN chỉ có ngày ở 41% số dòng, nên luồng của nó luôn phải dựng. PhreshPhish (Dalton và cộng sự, 2025; CC BY 4.0) ghi ngày thu
thập cho mọi dòng của cả hai lớp, nên thoạt nhìn có thể cho một thứ tự "thật" để so. Bước này đọc bốn cột `sha256`, `url`,
`label`, `date` của PhreshPhish tại một revision cố định trên Hugging Face. Parquet lưu từng cột riêng, nên cột HTML (phần lớn
trong khoảng 20 GB) không bị tải về. Sau đó:
- bỏ các URL mà `urllib` không phân tích được (một dòng);
- rút đúng 53.116 dòng, cỡ của PhishVN: các dòng có SHA-1 của `"sample:" + sha256` nhỏ nhất, nên cách rút cố định và không nhìn
  vào nhãn;
- tính 21 đặc trưng bằng bộ trích ở Bước 9a, xếp theo ngày thật, cắt 20 cửa sổ bằng nhau như PhishVN.

Kết quả bài đọc: tỷ lệ phishing theo cửa sổ dao động rất mạnh, có cửa sổ toàn phishing, vì benign được thu theo từng đợt (không có
dòng benign nào trong một chuỗi tháng liền). Thứ tự ngày của corpus này phản ánh lịch thu thập chứ không phải lúc trang xuất hiện,
nên mô hình học ở cửa sổ 1 không hề thấy một trang benign. Phép thử so luồng dựng với luồng thật vì vậy không chạy được.

Lần chạy đầu tải khoảng vài trăm MB (chỉ các cột cần dùng). Phép kiểm: `p5_phreshphish.json` từng byte.

In [ ]:
import pyarrow.parquet as pq
from huggingface_hub import HfFileSystem

PP_REPO = "datasets/phreshphish/phreshphish"
PP_REVISION = "eabec4b7a66324b79cc8a0ad856d1731dc26fe1a"   # v1.0.1, 2026-02-09
PP_CSV = WORK / "phreshphish_dated.csv"

if not PP_CSV.exists():
    fs = HfFileSystem()
    parts = []
    for f in sorted(fs.glob(f"{PP_REPO}@{PP_REVISION}/data/*.parquet")):
        with fs.open(f, "rb", block_size=1 << 20) as fh:
            t = pq.read_table(fh, columns=["sha256", "url", "label", "date"]).to_pandas()
        t["split"] = os.path.basename(f).split("-")[0]
        parts.append(t)
    pd.concat(parts, ignore_index=True).to_csv(PP_CSV, index=False)


def parses(url):
    try:
        urlparse(url if re.match(r"^[a-z]+://", url, re.I) else "http://" + url)
        return True
    except ValueError:
        return False


PP_FEATURES = ["url_len", "dom_len", "is_ip", "tld_len", "subdom_cnt", "letter_cnt", "digit_cnt",
               "special_cnt", "eq_cnt", "qm_cnt", "amp_cnt", "dot_cnt", "dash_cnt", "under_cnt",
               "letter_ratio", "digit_ratio", "spec_ratio", "slash_cnt", "entropy", "path_len", "query_len"]
PP_STREAM = "data/processed/p5/pp_stream.csv"

full = pd.read_csv(PP_CSV, low_memory=False)
ok = full["url"].astype(str).map(parses)
unparsable = int((~ok).sum())
full = full[ok].reset_index(drop=True)
draw = full["sha256"].astype(str).map(lambda k: hashlib.sha1(("sample:" + k).encode()).hexdigest())
src = full.loc[draw.sort_values(kind="stable").index[:53116]].sort_index().reset_index(drop=True)
pf = pd.DataFrame([extract(u) for u in src["url"].astype(str)])[PP_FEATURES]
pf["label"] = src["label"].map({"phish": "phishing", "benign": "benign"}).values
pf["collected_at"] = pd.to_datetime(src["date"]).dt.strftime("%Y-%m-%d").values
pf["id"] = src["sha256"].values
pf.to_csv(PP_STREAM, index=False)
tt = pd.to_datetime(pf["collected_at"])
bare = bare_host(src["url"])
PP_CORPUS = {"unparsable_dropped": unparsable, "corpus_rows": len(full),
             "corpus_phishing": int((full.label == "phish").sum()),
             "rows": len(pf), "phishing": int((pf.label == "phishing").sum()),
             "benign": int((pf.label == "benign").sum()),
             "first_date": f"{tt.min():%Y-%m-%d}", "last_date": f"{tt.max():%Y-%m-%d}",
             "distinct_dates": int(tt.nunique()), "duplicate_urls": int(len(bare) - bare.nunique())}
print(PP_CORPUS)

dpp, _ = load(PP_STREAM, spread_undated=False)
share = [float(dpp.y.iloc[i].mean()) for i in windows_of(dpp)]
print("phishing share per date-ordered window:", [round(x, 3) for x in share])

# the longest run of calendar months inside the span with no benign row (empty months count)
months = pd.period_range(dpp["_t"].min().to_period("M"), dpp["_t"].max().to_period("M"), freq="M")
has = set(dpp.loc[dpp.y == 0, "_t"].dt.to_period("M"))
best, cur = (None, None, 0), []
for m in months:
    cur = cur + [m] if m not in has else []
    if len(cur) > best[2]:
        best = (cur[0], cur[-1], len(cur))
print(f"no benign row from {best[0]} to {best[1]} ({best[2]} months)")

write_results(f"{RESULTS}/p5_phreshphish.json",
              {"corpus": PP_CORPUS, "window_phishing_share": share,
               "all_phishing_windows": sum(x == 1.0 for x in share),
               "benign_gap": {"first": str(best[0]), "last": str(best[1]), "months": best[2]}})
check_file("11 dated corpus", f"{RESULTS}/p5_phreshphish.json", "results/p5_phreshphish.json")
report("11 dated corpus")

## Bước 12: Tổng hợp

Bảng dưới gom mọi phép kiểm theo bước. Mỗi dòng là một file so từng byte, hoặc một phần của file JSON so từng giá trị.

In [ ]:
tbl = pd.DataFrame(CHECKS)
per_step = tbl.groupby("step", sort=False)["match"].agg(checks="size", matched="sum")
per_step["mismatched"] = per_step["checks"] - per_step["matched"]
display(per_step)
print(f"total: {int(tbl['match'].sum())}/{len(tbl)} checks match")
bad = tbl[~tbl["match"]]
if len(bad):
    display(bad)